# Saylani Advanced Python – 15 Tasks
Run each cell independently (Runtime ▸ Run all works too). Standard library only. Task 14 defaults to a demo; set `MODE = "cli"` for the interactive menu.

## Task 01 | Nested Transaction Analyzer

In [ ]:
"""Task 01 | Nested Transaction Analyzer"""
from collections import defaultdict


def analyze_transactions(transactions):
    if not isinstance(transactions, list):
        raise TypeError("transactions must be a list")
    seen = set()
    for t in transactions:
        missing = {"id", "customer", "category", "amount", "status"} - set(t)
        if missing:
            raise ValueError(f"Transaction {t} missing keys: {sorted(missing)}")
        if t["id"] in seen:
            raise ValueError(f"Duplicate transaction ID: {t['id']}")
        seen.add(t["id"])
        if not isinstance(t["amount"], (int, float)) or isinstance(t["amount"], bool) or t["amount"] < 0:
            raise ValueError(f"Invalid amount in transaction {t['id']}: {t['amount']!r}")

    totals = {t["customer"]: 0 for t in transactions}          # all customers appear
    by_cat = defaultdict(lambda: defaultdict(float))
    for t in transactions:
        if t["status"] == "success":
            totals[t["customer"]] += t["amount"]
            by_cat[t["customer"]][t["category"]] += t["amount"]

    def top_category(c):
        cats = by_cat.get(c)
        if not cats:
            return None
        return min(cats.items(), key=lambda kv: (-kv[1], kv[0]))[0]

    rows = sorted(totals.items(), key=lambda kv: (-kv[1], kv[0]))
    return [{"rank": i, "customer": c, "total": total, "top_category": top_category(c)}
            for i, (c, total) in enumerate(rows, 1)]


if __name__ == "__main__":
    data = [
        {"id": 1, "customer": "Ali", "category": "Food", "amount": 500, "status": "success"},
        {"id": 2, "customer": "Ali", "category": "Travel", "amount": 200, "status": "failed"},
        {"id": 3, "customer": "Sara", "category": "Food", "amount": 700, "status": "success"},
        {"id": 4, "customer": "Zoya", "category": "Food", "amount": 50, "status": "failed"},
    ]
    for row in analyze_transactions(data):
        print(row)
    try:
        analyze_transactions(data + [data[0]])
    except ValueError as e:
        print("Rejected:", e)


## Task 02 | Longest Valid Window (sliding window, O(n))

In [ ]:
"""Task 02 | Longest Valid Window (sliding window, O(n))"""


def longest_unique_window(text):
    if not isinstance(text, str):
        raise TypeError("text must be a string")
    last_seen = {}
    start = best_start = best_len = 0
    for i, ch in enumerate(text):
        if ch in last_seen and last_seen[ch] >= start:
            start = last_seen[ch] + 1
        last_seen[ch] = i
        if i - start + 1 > best_len:          # strict '>' keeps the earliest window
            best_len, best_start = i - start + 1, start
    return text[best_start:best_start + best_len], best_start, best_len


if __name__ == "__main__":
    for s in ["abcabcbb", "", "bbbbb", "pwwkew", "a b!a b"]:
        print(repr(s), "->", longest_unique_window(s))


## Task 03 | Recursive JSON Path Explorer

In [ ]:
"""Task 03 | Recursive JSON Path Explorer"""


def _equal(value, target):
    # bool must not equal numeric 1/0 (True == 1 in Python)
    if isinstance(value, bool) != isinstance(target, bool):
        return False
    return value == target


def find_paths(data, target):
    results = []

    def walk(node, path):
        if isinstance(node, dict):
            for k, v in node.items():
                walk(v, f"{path}.{k}" if path else str(k))
        elif isinstance(node, list):
            for i, v in enumerate(node):
                walk(v, f"{path}[{i}]")
        elif _equal(node, target):
            results.append(path)

    walk(data, "")
    return results


if __name__ == "__main__":
    print(find_paths({"a": [{"x": 5}, 5], "b": {"y": 5}}, 5))
    print(find_paths({"flag": True, "n": 1, "l": [1, True]}, 1))
    print(find_paths({"flag": True, "n": 1, "l": [1, True]}, True))
    print(find_paths([], 5))


## Task 04 | Dependency Execution Planner (Kahn's algorithm, alphabetical tie-break)

In [ ]:
"""Task 04 | Dependency Execution Planner (Kahn's algorithm, alphabetical tie-break)"""
import heapq


def plan_execution(tasks, dependencies):
    tasks = list(tasks)
    if len(set(tasks)) != len(tasks):
        raise ValueError("Duplicate task names provided")
    known = set(tasks)
    graph = {t: set() for t in tasks}
    indeg = {t: 0 for t in tasks}
    for pair in dependencies:
        if len(pair) != 2:
            raise ValueError(f"Invalid dependency pair: {pair!r}")
        a, b = pair
        for name in (a, b):
            if name not in known:
                raise ValueError(f"Unknown task name: {name!r}")
        if a == b:
            raise ValueError(f"Task {a!r} cannot depend on itself")
        if b not in graph[a]:
            graph[a].add(b)
            indeg[b] += 1

    heap = [t for t in tasks if indeg[t] == 0]
    heapq.heapify(heap)
    order = []
    while heap:
        t = heapq.heappop(heap)
        order.append(t)
        for nxt in graph[t]:
            indeg[nxt] -= 1
            if indeg[nxt] == 0:
                heapq.heappush(heap, nxt)
    if len(order) != len(tasks):
        stuck = sorted(t for t in tasks if t not in order)
        raise ValueError(f"Cycle detected among tasks: {stuck}")
    return order


if __name__ == "__main__":
    print(plan_execution(["build", "test", "deploy"], [("build", "test"), ("test", "deploy")]))
    print(plan_execution(["c", "b", "a"], []))
    for tasks, deps in [(["a", "b"], [("a", "b"), ("b", "a")]), (["a"], [("a", "zzz")])]:
        try:
            plan_execution(tasks, deps)
        except ValueError as e:
            print("Error:", e)


## Task 05 | Expression Validator and Evaluator (shunting-yard, no eval)

In [ ]:
"""Task 05 | Expression Validator and Evaluator (shunting-yard, no eval)"""
import re

_TOKEN = re.compile(r"\s*(?:(\d+)|(.))")


def tokenize(expr):
    if not isinstance(expr, str):
        raise TypeError("expression must be a string")
    tokens, pos = [], 0
    expr = expr.strip()
    if not expr:
        raise ValueError("Empty expression")
    while pos < len(expr):
        m = _TOKEN.match(expr, pos)
        num, op = m.groups()
        if num is not None:
            tokens.append(("num", int(num)))
        elif op in "+-*/()":
            tokens.append(("op", op))
        else:
            raise ValueError(f"Invalid character {op!r} at position {m.start(2)}")
        pos = m.end()
    return tokens


PREC = {"+": 1, "-": 1, "*": 2, "/": 2, "u-": 3}


def _apply(op, vals):
    if op == "u-":
        if not vals:
            raise ValueError("Malformed expression")
        vals.append(-vals.pop())
        return
    if len(vals) < 2:
        raise ValueError("Malformed expression")
    b, a = vals.pop(), vals.pop()
    if op == "+": vals.append(a + b)
    elif op == "-": vals.append(a - b)
    elif op == "*": vals.append(a * b)
    else:
        if b == 0:
            raise ZeroDivisionError("Division by zero")
        vals.append(a / b)


def evaluate(expr):
    vals, ops = [], []
    prev = None  # 'num', ')', or other -> decides unary vs binary minus
    for kind, v in tokenize(expr):
        if kind == "num":
            if prev in ("num", ")"):
                raise ValueError("Malformed expression: missing operator")
            vals.append(v); prev = "num"
        elif v == "(":
            if prev in ("num", ")"):
                raise ValueError("Malformed expression: missing operator before '('")
            ops.append(v); prev = "("
        elif v == ")":
            if prev not in ("num", ")"):
                raise ValueError("Malformed expression: unexpected ')'")
            while ops and ops[-1] != "(":
                _apply(ops.pop(), vals)
            if not ops:
                raise ValueError("Mismatched parentheses")
            ops.pop(); prev = ")"
        else:
            if prev in ("num", ")"):
                op = v
            elif v == "-":
                op = "u-"
            elif v == "+":
                prev = "op"; continue          # unary plus: ignore
            else:
                raise ValueError(f"Malformed expression: unexpected {v!r}")
            if op != "u-":                      # unary ops are right-assoc
                while ops and ops[-1] != "(" and PREC[ops[-1]] >= PREC[op]:
                    _apply(ops.pop(), vals)
            ops.append(op); prev = "op"
    if prev not in ("num", ")"):
        raise ValueError("Malformed expression: ends with operator")
    while ops:
        op = ops.pop()
        if op == "(":
            raise ValueError("Mismatched parentheses")
        _apply(op, vals)
    if len(vals) != 1:
        raise ValueError("Malformed expression")
    return float(vals[0])


if __name__ == "__main__":
    tests = ["-(3 + 2) * 4 + 18 / 3", "2 + 3 * 4", "(2 + 3) * 4", "--5", "10 / (5 - 5)",
             "2 +", "(1 + 2", "3 $ 4", "", "2 (3)", "4 - -2"]
    for t in tests:
        try:
            print(f"{t!r:28} = {evaluate(t)}")
        except (ValueError, ZeroDivisionError) as e:
            print(f"{t!r:28} -> {type(e).__name__}: {e}")


## Task 06 | Banking Ledger with OOP

In [ ]:
"""Task 06 | Banking Ledger with OOP"""
from dataclasses import dataclass
from itertools import count


@dataclass(frozen=True)
class Transaction:
    txn_id: int
    action: str
    amount: float
    balance_after: float


class Account:
    def __init__(self, account_id, owner, opening_balance=0, _ids=None):
        if opening_balance < 0:
            raise ValueError("Opening balance cannot be negative")
        self._id, self._owner, self._balance = account_id, owner, opening_balance
        self._history = []
        self._ids = _ids or count(1)

    @property
    def account_id(self): return self._id
    @property
    def owner(self): return self._owner
    @property
    def balance(self): return self._balance
    @property
    def history(self): return tuple(self._history)

    @staticmethod
    def _check_amount(amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)) or amount <= 0:
            raise ValueError("Amount must be a positive number")

    def _record(self, action, amount):
        self._history.append(Transaction(next(self._ids), action, amount, self._balance))

    def deposit(self, amount, action="DEPOSIT"):
        self._check_amount(amount)
        self._balance += amount
        self._record(action, amount)

    def withdraw(self, amount, action="WITHDRAWAL"):
        self._check_amount(amount)
        if amount > self._balance:
            raise ValueError("Insufficient funds")
        self._balance -= amount
        self._record(action, amount)

    def statement(self):
        lines = [f"Statement for {self._owner} ({self._id})",
                 f"{'ID':<5}{'Action':<16}{'Amount':>10}{'Balance':>12}"]
        lines += [f"{t.txn_id:<5}{t.action:<16}{t.amount:>10.2f}{t.balance_after:>12.2f}" for t in self._history]
        return "\n".join(lines)


class Bank:
    def __init__(self):
        self._accounts = {}
        self._ids = count(1)       # bank-wide unique transaction ids

    def open_account(self, account_id, owner, opening_balance=0):
        if account_id in self._accounts:
            raise ValueError(f"Account {account_id} already exists")
        acc = Account(account_id, owner, opening_balance, self._ids)
        if opening_balance:
            acc._record("OPENING", opening_balance)
        self._accounts[account_id] = acc
        return acc

    def get(self, account_id):
        try:
            return self._accounts[account_id]
        except KeyError:
            raise KeyError(f"Account {account_id} does not exist") from None

    def transfer(self, src_id, dst_id, amount):
        src, dst = self.get(src_id), self.get(dst_id)
        if src is dst:
            raise ValueError("Cannot transfer to the same account")
        Account._check_amount(amount)
        if amount > src.balance:                  # validate everything before mutating -> atomic
            raise ValueError("Insufficient funds")
        src.withdraw(amount, f"TRANSFER_OUT->{dst_id}")
        dst.deposit(amount, f"TRANSFER_IN<-{src_id}")


if __name__ == "__main__":
    bank = Bank()
    bank.open_account("A", "Ali", 1000)
    bank.open_account("B", "Sara", 200)
    bank.transfer("A", "B", 350)
    print(bank.get("A").balance, bank.get("B").balance)   # 650 550
    print(bank.get("A").statement()); print(bank.get("B").statement())
    for bad in [lambda: bank.transfer("A", "B", 5000), lambda: bank.transfer("A", "Z", 10),
                lambda: bank.get("A").deposit(-5)]:
        try:
            bad()
        except (ValueError, KeyError) as e:
            print("Rejected:", e)
    print("Balances unchanged:", bank.get("A").balance, bank.get("B").balance)
    try:
        bank.get("A").history[0].amount = 1
    except Exception as e:
        print("Immutable record:", type(e).__name__)


## Task 07 | Employee Payroll Engine

In [ ]:
"""Task 07 | Employee Payroll Engine"""
from abc import ABC, abstractmethod


class Employee(ABC):
    def __init__(self, name):
        if not name or not isinstance(name, str):
            raise ValueError("Employee name is required")
        self.name = name

    @abstractmethod
    def calculate_pay(self): ...

    def __repr__(self):
        return f"{type(self).__name__}({self.name!r})"


def _non_negative(value, label):
    if isinstance(value, bool) or not isinstance(value, (int, float)) or value < 0:
        raise ValueError(f"{label} must be a non-negative number")
    return value


class SalariedEmployee(Employee):
    def __init__(self, name, monthly_salary):
        super().__init__(name)
        self.monthly_salary = _non_negative(monthly_salary, "monthly_salary")

    def calculate_pay(self):
        return self.monthly_salary


class HourlyEmployee(Employee):
    OVERTIME_AFTER, OVERTIME_RATE = 40, 1.5

    def __init__(self, name, hourly_rate, hours_worked):
        super().__init__(name)
        self.hourly_rate = _non_negative(hourly_rate, "hourly_rate")
        self.hours_worked = _non_negative(hours_worked, "hours_worked")

    def calculate_pay(self):
        regular = min(self.hours_worked, self.OVERTIME_AFTER)
        overtime = max(self.hours_worked - self.OVERTIME_AFTER, 0)
        return self.hourly_rate * (regular + overtime * self.OVERTIME_RATE)


class CommissionEmployee(Employee):
    def __init__(self, name, base, sales, rate):
        super().__init__(name)
        self.base = _non_negative(base, "base")
        self.sales = _non_negative(sales, "sales")
        if not 0 <= rate <= 1:
            raise ValueError("rate must be between 0 and 1")
        self.rate = rate

    def calculate_pay(self):
        return self.base + self.sales * self.rate


def payroll_summary(employees):
    rows = [(e.name, type(e).__name__, e.calculate_pay()) for e in employees]
    lines = [f"{'Name':<10}{'Type':<22}{'Pay':>12}"]
    lines += [f"{n:<10}{t:<22}{p:>12,.2f}" for n, t, p in rows]
    lines.append(f"{'TOTAL':<32}{sum(p for *_, p in rows):>12,.2f}")
    return "\n".join(lines)


if __name__ == "__main__":
    staff = [SalariedEmployee("Ayesha", 80000), HourlyEmployee("Bilal", 20, 45),
             CommissionEmployee("Hina", 500, 4000, 0.10)]
    print(payroll_summary(staff))
    try:
        Employee("x")
    except TypeError as e:
        print("Blocked:", e)


## Task 08 | Cache Decorator with Expiration

In [ ]:
"""Task 08 | Cache Decorator with Expiration

Behavior for unhashable inputs: arguments that cannot be hashed (list, dict, set...)
are NOT cacheable, so the function is called directly and counted as a miss
("bypass"); no exception is raised.
Keyword order does not matter: kwargs are sorted by name to build the key.
"""
import functools
import time


def ttl_cache(seconds):
    if seconds <= 0:
        raise ValueError("seconds must be positive")

    def decorator(func):
        cache = {}
        stats = {"hits": 0, "misses": 0}

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            key = (args, tuple(sorted(kwargs.items())))
            try:
                hash(key)
            except TypeError:
                stats["misses"] += 1
                return func(*args, **kwargs)
            now = time.monotonic()
            entry = cache.get(key)
            if entry is not None and now - entry[1] < seconds:
                stats["hits"] += 1
                return entry[0]
            stats["misses"] += 1
            value = func(*args, **kwargs)
            cache[key] = (value, time.monotonic())
            return value

        wrapper.cache_info = lambda: dict(stats, size=len(cache))
        wrapper.cache_clear = lambda: (cache.clear(), stats.update(hits=0, misses=0))[0]
        return wrapper
    return decorator


if __name__ == "__main__":
    @ttl_cache(1)
    def add(x, y):
        """Add two numbers."""
        return x + y

    print(add(x=2, y=3), add(y=3, x=2), add.cache_info())   # 5 5, 1 hit 1 miss
    time.sleep(1.1)
    add(x=2, y=3)
    print("After expiry:", add.cache_info())                # 2 misses
    @ttl_cache(5)
    def total(items): return sum(items)
    total([1, 2]); total([1, 2])
    print("Unhashable:", total.cache_info())
    print(add.__name__, "|", add.__doc__)


## Task 09 | Streaming Log Inspector

In [ ]:
"""Task 09 | Streaming Log Inspector"""
import os
import re
import tempfile
from collections import Counter
from datetime import datetime

_LINE = re.compile(r"^(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}) \| ([A-Za-z]+) \| (.*)$")


def _parse(line):
    m = _LINE.match(line.rstrip("\n"))
    if not m:
        return None
    try:
        datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S")
    except ValueError:
        return None
    return m.groups()


def read_errors(path, stats=None):
    """Yield (line_number, timestamp, message) for ERROR lines. stats['malformed'] counts bad lines."""
    stats = stats if stats is not None else {}
    stats.setdefault("malformed", 0)
    with open(path, encoding="utf-8") as fh:       # iterates lazily, one line at a time
        for n, line in enumerate(fh, 1):
            if not line.strip():
                continue
            parsed = _parse(line)
            if parsed is None:
                stats["malformed"] += 1
                continue
            ts, level, msg = parsed
            if level.upper() == "ERROR":
                yield n, ts, msg


def summarize_errors(path):
    stats = {}
    counts = Counter(ts[:10] for _, ts, _ in read_errors(path, stats))
    return dict(sorted(counts.items())), stats["malformed"]


SAMPLE = """2026-11-07 10:30:00 | ERROR | Connection lost
2026-11-07 10:31:00 | INFO | Reconnected
this line is garbage
2026-11-07 11:00:00 | ERROR | Disk almost full
2026-11-08 09:00:00 | ERROR | Timeout
2026-13-45 99:99:99 | ERROR | Bad date
"""

if __name__ == "__main__":
    path = os.path.join(tempfile.gettempdir(), "sample.log")
    with open(path, "w") as f:
        f.write(SAMPLE)
    for rec in read_errors(path):
        print(rec)
    print("Summary (counts by date, malformed):", summarize_errors(path))


## Task 10 | CSV Inventory Reconciliation (generates its own sample data)

In [ ]:
"""Task 10 | CSV Inventory Reconciliation (generates its own sample data)"""
import csv
import os


def reconcile(inventory_path="inventory.csv", movements_path="movements.csv",
              out_path="updated_inventory.csv", rejected_path="rejected_movements.csv"):
    stock, names, order = {}, {}, []
    with open(inventory_path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            sku = row["sku"].strip()
            names[sku] = row["name"].strip()
            stock[sku] = int(row["quantity"])
            order.append(sku)

    rejected = []
    with open(movements_path, newline="", encoding="utf-8") as f:
        for line_no, row in enumerate(csv.DictReader(f), start=2):
            sku, mtype = (row.get("sku") or "").strip(), (row.get("type") or "").strip().upper()
            reason = None
            try:
                units = int(row.get("units"))
            except (TypeError, ValueError):
                units, reason = None, "units not an integer"
            if reason is None:
                if sku not in stock: reason = "unknown SKU"
                elif mtype not in ("IN", "OUT"): reason = "invalid movement type"
                elif units <= 0: reason = "units must be positive"
                elif mtype == "OUT" and stock[sku] - units < 0: reason = "insufficient stock"
            if reason:
                rejected.append({"line": line_no, "sku": sku, "type": row.get("type"),
                                 "units": row.get("units"), "reason": reason})
            else:
                stock[sku] += units if mtype == "IN" else -units

    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f); w.writerow(["sku", "name", "quantity"])
        for sku in order:
            w.writerow([sku, names[sku], stock[sku]])
    with open(rejected_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["line", "sku", "type", "units", "reason"])
        w.writeheader(); w.writerows(rejected)
    return stock, rejected


def make_sample():
    with open("inventory.csv", "w", newline="") as f:
        f.write("sku,name,quantity\nP01,Pen,10\nP02,Notebook,25\n")
    with open("movements.csv", "w", newline="") as f:
        f.write("sku,type,units\nP01,OUT,4\nP01,OUT,9\nP01,IN,2\nP99,IN,5\nP02,IN,0\nP02,SELL,3\nP02,OUT,-1\n")


if __name__ == "__main__":
    make_sample()
    stock, rejected = reconcile()
    print("Final stock:", stock)            # P01 = 8
    for r in rejected:
        print("Rejected:", r)
    print(open("updated_inventory.csv").read())
    print(open("rejected_movements.csv").read())


## Task 11 | Multi-File Student Gradebook (generates its own sample data)

In [ ]:
"""Task 11 | Multi-File Student Gradebook (generates its own sample data)"""
import csv


def grade_for(avg):
    for cutoff, g in ((85, "A"), (70, "B"), (55, "C"), (40, "D")):
        if avg >= cutoff:
            return g
    return "F"


def build_report(students_path="students.csv", scores_path="scores.csv", out_path="report.csv"):
    with open(students_path, newline="", encoding="utf-8") as f:
        students = {r["student_id"].strip(): r["name"].strip() for r in csv.DictReader(f)}
    marks, seen, errors = {sid: [] for sid in students}, set(), []
    with open(scores_path, newline="", encoding="utf-8") as f:
        for n, r in enumerate(csv.DictReader(f), start=2):
            sid, subj = r["student_id"].strip(), r["subject"].strip()
            try:
                m = float(r["marks"])
            except (TypeError, ValueError):
                errors.append((n, "marks not numeric")); continue
            if sid not in students: errors.append((n, f"unknown student {sid}")); continue
            if not 0 <= m <= 100: errors.append((n, f"marks {m} out of range")); continue
            if (sid, subj) in seen: errors.append((n, f"duplicate {sid}/{subj}")); continue
            seen.add((sid, subj)); marks[sid].append(m)

    rows = []
    for sid, name in students.items():
        ms = marks[sid]
        avg = round(sum(ms) / len(ms), 2) if ms else 0.0
        rows.append({"student_id": sid, "name": name, "subjects": len(ms),
                     "average": avg, "grade": grade_for(avg) if ms else "N/A"})
    rows.sort(key=lambda r: (-r["average"], r["student_id"]))
    for i, r in enumerate(rows, 1):
        r["rank"] = i
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["rank", "student_id", "name", "subjects", "average", "grade"])
        w.writeheader(); w.writerows(rows)
    return rows, errors


def make_sample():
    open("students.csv", "w").write("student_id,name\n101,Ali\n102,Sara\n103,Bilal\n")
    open("scores.csv", "w").write(
        "student_id,subject,marks\n101,Math,90\n101,Physics,80\n101,English,85\n"
        "102,Math,95\n102,Physics,60\n102,Math,70\n999,Math,50\n103,Math,120\n103,English,38\n")


if __name__ == "__main__":
    make_sample()
    rows, errors = build_report()
    for r in rows:
        print(r)
    print("Rejected rows:", errors)


## Task 12 | Custom Iterable Range Toolkit

In [ ]:
"""Task 12 | Custom Iterable Range Toolkit"""


class ReversibleRange:
    def __init__(self, start, stop, step=1):
        for v in (start, stop, step):
            if isinstance(v, bool) or not isinstance(v, int):
                raise TypeError("start, stop and step must be integers")
        if step == 0:
            raise ValueError("step must not be zero")
        self.start, self.stop, self.step = start, stop, step

    def __len__(self):
        if self.step > 0:
            return max(0, (self.stop - self.start + self.step - 1) // self.step)
        return max(0, (self.start - self.stop - self.step - 1) // -self.step)

    def __iter__(self):
        cur = self.start
        while (cur < self.stop) if self.step > 0 else (cur > self.stop):
            yield cur
            cur += self.step

    def __reversed__(self):
        n = len(self)
        for i in range(n - 1, -1, -1):
            yield self.start + i * self.step

    def __contains__(self, x):
        if isinstance(x, bool) or not isinstance(x, int):
            return False
        offset = x - self.start
        if offset % self.step:
            return False
        return 0 <= offset // self.step < len(self)

    def __repr__(self):
        return f"ReversibleRange({self.start}, {self.stop}, {self.step})"


if __name__ == "__main__":
    r = ReversibleRange(2, 11, 3)
    print(list(r), list(reversed(r)), len(r), 5 in r, 6 in r)
    # Verify against built-in range
    import itertools
    for a, b, s in itertools.product(range(-6, 7), range(-6, 7), [-3, -2, -1, 1, 2, 3]):
        mine, ref = ReversibleRange(a, b, s), range(a, b, s)
        assert list(mine) == list(ref) and list(reversed(mine)) == list(reversed(ref)) and len(mine) == len(ref)
        assert all((x in mine) == (x in ref) for x in range(-8, 9))
    print("Matches built-in range for all tested combinations.")


## Task 13 | Booking Conflict Scheduler

In [ ]:
"""Task 13 | Booking Conflict Scheduler"""
import bisect
from dataclasses import dataclass
from datetime import datetime


@dataclass(frozen=True)
class Booking:
    booking_id: str
    room: str
    start: datetime
    end: datetime


class Scheduler:
    def __init__(self):
        self._by_room = {}       # room -> list of Booking sorted by start
        self._ids = set()

    def book(self, booking_id, room, start, end):
        """Return (accepted: bool, conflicts: list[Booking] | reason: str)."""
        if not (isinstance(start, datetime) and isinstance(end, datetime)):
            raise TypeError("start and end must be datetime objects")
        if end <= start:
            return False, "end must be after start"
        if booking_id in self._ids:
            return False, f"duplicate booking id {booking_id!r}"
        bookings = self._by_room.setdefault(room, [])
        conflicts = [b for b in bookings if start < b.end and b.start < end]  # adjacent is fine
        if conflicts:
            return False, conflicts
        new = Booking(booking_id, room, start, end)
        bisect.insort(bookings, new, key=lambda b: b.start)       # Python 3.10+
        self._ids.add(booking_id)
        return True, []

    def bookings(self, room):
        return list(self._by_room.get(room, []))


if __name__ == "__main__":
    d = lambda h, m=0: datetime(2026, 11, 7, h, m)
    s = Scheduler()
    print(s.book("B1", "A", d(9), d(10)))
    print(s.book("B2", "A", d(10), d(11)))
    ok, conflicts = s.book("B3", "A", d(9, 30), d(10, 30))
    print("B3 accepted?", ok, "| conflicts:", [c.booking_id for c in conflicts])
    print(s.book("B4", "B", d(9, 30), d(10, 30)))     # different room OK
    print(s.book("B5", "A", d(12), d(11)))            # invalid
    print(s.book("B1", "A", d(13), d(14)))            # duplicate id


## Task 14 | Command-Line Contact Manager (storage + logic + CLI layers in one file)

In [ ]:
"""Task 14 | Command-Line Contact Manager (storage + logic + CLI layers in one file)

Colab: input() works in a cell, but the default is a scripted DEMO. Set MODE = "cli" to use the menu.
"""
import json
import os
import re
import tempfile

# ---------------- storage layer ----------------
class StorageError(Exception):
    pass


def load_contacts(path):
    if not os.path.exists(path):
        return []
    try:
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        if not isinstance(data, list):
            raise ValueError("root must be a list")
        return data
    except (json.JSONDecodeError, ValueError) as e:
        backup = path + ".corrupt"            # never silently destroy existing data
        os.replace(path, backup)
        raise StorageError(f"Corrupt file moved to {backup}: {e}") from e


def save_contacts(path, contacts):
    tmp = path + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(contacts, f, indent=2)
    os.replace(tmp, path)                     # atomic write


# ---------------- business logic ----------------
EMAIL = re.compile(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
PHONE = re.compile(r"^\+?[0-9\- ]{7,15}$")


def validate(name, phone, email):
    if not name or not name.strip(): raise ValueError("Name is required")
    if phone and not PHONE.match(phone): raise ValueError("Invalid phone number")
    if email and not EMAIL.match(email): raise ValueError("Invalid email")


def add_contact(contacts, contact_id, name, phone="", email=""):
    validate(name, phone, email)
    if any(c["id"] == contact_id for c in contacts):
        raise ValueError(f"ID {contact_id} already exists")
    contacts.append({"id": contact_id, "name": name.strip(), "phone": phone, "email": email})


def search_contacts(contacts, query):
    q = query.strip().lower()
    return [c for c in contacts if q in c["name"].lower()]


def update_contact(contacts, contact_id, **fields):
    for c in contacts:
        if c["id"] == contact_id:
            new = {**c, **{k: v for k, v in fields.items() if v is not None and k in ("name", "phone", "email")}}
            validate(new["name"], new["phone"], new["email"])
            c.update(new)
            return c
    raise KeyError(f"No contact with ID {contact_id}")


def delete_contact(contacts, contact_id):
    for i, c in enumerate(contacts):
        if c["id"] == contact_id:
            return contacts.pop(i)
    raise KeyError(f"No contact with ID {contact_id}")


# ---------------- CLI layer ----------------
def cli(path="contacts.json"):
    try:
        contacts = load_contacts(path)
    except StorageError as e:
        print("WARNING:", e); contacts = []
    menu = "\n1.Add 2.Search 3.Update 4.Delete 5.List 0.Exit\n> "
    while (choice := input(menu).strip()) != "0":
        try:
            if choice == "1":
                add_contact(contacts, int(input("ID: ")), input("Name: "), input("Phone: "), input("Email: "))
            elif choice == "2":
                print(*search_contacts(contacts, input("Search: ")), sep="\n")
            elif choice == "3":
                cid = int(input("ID: "))
                update_contact(contacts, cid, name=input("New name (blank=keep): ") or None,
                               phone=input("New phone: ") or None, email=input("New email: ") or None)
            elif choice == "4":
                delete_contact(contacts, int(input("ID: ")))
            elif choice == "5":
                print(*contacts, sep="\n")
            else:
                print("Invalid choice"); continue
            save_contacts(path, contacts)
        except (ValueError, KeyError) as e:
            print("Error:", e)


def demo():
    path = os.path.join(tempfile.gettempdir(), "contacts_demo.json")
    if os.path.exists(path): os.remove(path)
    c = load_contacts(path)
    add_contact(c, 1, "Ayesha", "0300-1234567", "ayesha@example.com")
    add_contact(c, 2, "Bilal", "03111234567")
    save_contacts(path, c)
    c2 = load_contacts(path)                                   # "restart"
    print("Search 'aye' after restart:", search_contacts(c2, "aye"))
    update_contact(c2, 2, phone="03219998888"); delete_contact(c2, 1)
    print("List:", c2)
    for bad in (lambda: add_contact(c2, 2, "Dup"), lambda: add_contact(c2, 3, "X", email="bad")):
        try: bad()
        except ValueError as e: print("Rejected:", e)
    open(path, "w").write("{not json")                          # corrupt file test
    try: load_contacts(path)
    except StorageError as e: print("Safe handling:", e)


MODE = "demo"   # change to "cli" for the interactive menu
if __name__ == "__main__":
    demo() if MODE == "demo" else cli()


## Task 15 | Integrated Order Fulfillment System (includes unittest tests)

In [ ]:
"""Task 15 | Integrated Order Fulfillment System (includes unittest tests)"""
import json
import os
import tempfile
import unittest
import uuid


class OrderError(Exception):
    pass


class Product:
    def __init__(self, pid, name, price, stock):
        if price < 0 or stock < 0:
            raise ValueError("price and stock must be non-negative")
        self.pid, self.name, self.price, self.stock = pid, name, price, stock


class Inventory:
    def __init__(self, products):
        self._p = {p.pid: p for p in products}

    @classmethod
    def from_json(cls, path):
        with open(path, encoding="utf-8") as f:
            return cls(Product(d["id"], d["name"], d["price"], d["stock"]) for d in json.load(f))

    def get(self, pid):
        if pid not in self._p:
            raise OrderError(f"Unknown product {pid!r}")
        return self._p[pid]

    def stock_of(self, pid):
        return self.get(pid).stock

    def validate(self, items):
        for pid, qty in items.items():
            p = self.get(pid)
            if qty > p.stock:
                raise OrderError(f"Out of stock: {pid} (requested {qty}, available {p.stock})")

    def deduct(self, items):
        for pid, qty in items.items():
            self._p[pid].stock -= qty

    def restore(self, items):
        for pid, qty in items.items():
            self._p[pid].stock += qty


class Cart:
    def __init__(self, inventory):
        self._inv, self.items, self.discount_pct = inventory, {}, 0

    def add(self, pid, qty):
        if isinstance(qty, bool) or not isinstance(qty, int) or qty <= 0:
            raise OrderError("Quantity must be a positive integer")
        self._inv.get(pid)
        self.items[pid] = self.items.get(pid, 0) + qty

    def apply_discount(self, pct):
        if not 0 <= pct <= 100:
            raise OrderError("Discount must be between 0 and 100")
        self.discount_pct = pct

    def subtotal(self):
        return sum(self._inv.get(pid).price * q for pid, q in self.items.items())

    def total(self):
        return round(self.subtotal() * (1 - self.discount_pct / 100), 2)


class Order:
    def __init__(self, items, total, discount_pct):
        self.order_id = uuid.uuid4().hex[:12]
        self.items, self.total, self.discount_pct = dict(items), total, discount_pct

    def to_dict(self):
        return {"order_id": self.order_id, "items": self.items,
                "discount_pct": self.discount_pct, "total": self.total}


class OrderService:
    def __init__(self, inventory, orders_path):
        self.inventory, self.orders_path = inventory, orders_path

    def _load(self):
        if not os.path.exists(self.orders_path):
            return []
        with open(self.orders_path, encoding="utf-8") as f:
            return json.load(f)

    def _persist(self, orders):
        tmp = self.orders_path + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(orders, f, indent=2)
        os.replace(tmp, self.orders_path)

    def place_order(self, cart):
        if not cart.items:
            raise OrderError("Cart is empty")
        self.inventory.validate(cart.items)                   # validate everything first
        order = Order(cart.items, cart.total(), cart.discount_pct)
        self.inventory.deduct(cart.items)
        try:
            self._persist(self._load() + [order.to_dict()])
        except Exception as e:                                # roll back stock
            self.inventory.restore(cart.items)
            raise OrderError(f"Could not save order, stock restored: {e}") from e
        return order


# ---------------- unit tests ----------------
class OrderTests(unittest.TestCase):
    def setUp(self):
        self.dir = tempfile.mkdtemp()
        self.inv = Inventory([Product("P1", "Pen", 100, 5)])
        self.svc = OrderService(self.inv, os.path.join(self.dir, "orders.json"))

    def test_successful_order_with_discount(self):
        cart = Cart(self.inv); cart.add("P1", 2); cart.apply_discount(10)
        order = self.svc.place_order(cart)
        self.assertEqual(order.total, 180)
        self.assertEqual(self.inv.stock_of("P1"), 3)

    def test_out_of_stock_and_invalid_qty(self):
        cart = Cart(self.inv); cart.add("P1", 9)
        with self.assertRaises(OrderError): self.svc.place_order(cart)
        self.assertEqual(self.inv.stock_of("P1"), 5)
        with self.assertRaises(OrderError): Cart(self.inv).add("P1", 0)

    def test_rollback_on_persistence_failure(self):
        bad = OrderService(self.inv, os.path.join(self.dir, "missing_dir", "orders.json"))
        cart = Cart(self.inv); cart.add("P1", 2)
        with self.assertRaises(OrderError): bad.place_order(cart)
        self.assertEqual(self.inv.stock_of("P1"), 5)

    def test_unique_order_ids(self):
        ids = set()
        for _ in range(3):
            cart = Cart(self.inv); cart.add("P1", 1)
            ids.add(self.svc.place_order(cart).order_id)
        self.assertEqual(len(ids), 3)


def demo():
    d = tempfile.mkdtemp()
    catalog = os.path.join(d, "catalog.json")
    json.dump([{"id": "P1", "name": "Pen", "price": 100, "stock": 5}], open(catalog, "w"))
    inv = Inventory.from_json(catalog)
    svc = OrderService(inv, os.path.join(d, "orders.json"))
    cart = Cart(inv); cart.add("P1", 2); cart.apply_discount(10)
    order = svc.place_order(cart)
    print("Order:", order.to_dict(), "| stock now:", inv.stock_of("P1"))
    print("Saved:", open(svc.orders_path).read())


if __name__ == "__main__":
    demo()
    # argv=[''] + exit=False makes unittest work inside Colab/Jupyter too
    unittest.main(argv=["ignored"], exit=False, verbosity=2)
